# Test on ERA5

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import os

import rasterio as rio
from pyproj import CRS

In [ ]:
from etdataset import era5
from etdataset.dem import get_dem_from_roi
from etdataset.era5 import download, prepare_data, read
from etdataset.era5_type import ERA5Dataset, ERA5Var
from etdataset.interpolation import create_grid_dataset
from etdataset.plot import plot_dataset, plot_dem
from etdataset.temperature import RescalTempMethod
from etdataset.utils import get_utm_bbox_from_roi, read_product_list


In [ ]:
import logging

from etdataset.logging import LoggerManager

LoggerManager.set_level(logging.DEBUG)

## Inputs

In [ ]:
product_list = read_product_list(
    "landsat_products.csv", CRS(4326), "overlap_geometry"
)

In [ ]:
product_list

In [ ]:
output = "out"
os.makedirs(output, exist_ok=True)

## Download required data

In [ ]:
bounds = rio.coords.BoundingBox(*product_list.loc[0].geometry.bounds)
crs = CRS(4326)
bounds

In [ ]:
date = dt.datetime(2023, 3, 3, 10, 30)

Download ERA5 dataset 

In [ ]:
download(
    date=date,
    dataset=ERA5Dataset.ERA5,
    path=os.path.join(output, "ERA5_data"),
)

In [ ]:
download(
    date=date,
    dataset=ERA5Dataset.ERA5LAND,
    path=os.path.join(output, "ERA5_data"),
)

## Read ERA5 data

In [ ]:
raw_data = read(
    product=os.path.join(output, "ERA5_data", "download_era5_2023-03-03.zip"),
)

In [ ]:
raw_data.sizes == {"time": 23, "latitude": 721, "longitude": 1440}

In [ ]:
diff = (
    raw_data[ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD_CLEAR_SKY.key]
    - raw_data[ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD.key]
)
diff.plot()

## Add ERA5 data

Artificial data

In [ ]:
# Create a test dataset
grid = create_grid_dataset(bounds, crs, 0.1)
grid["height"] = grid["grid"].copy()
# Add attributes
grid.attrs["vis_date"] = date.date()
grid.attrs["vis_time"] = date.time()
grid.attrs["tir_date"] = date.date()
grid.attrs["tir_time"] = date.time()
grid

In [ ]:
data_era5 = era5.add(
    data=grid,
    dataset=ERA5Dataset.ERA5,
    path=os.path.join(output, "ERA5_data"),
)

In [ ]:
data_era5.data_vars

In [ ]:
data_era5.rsd_era5.plot()

In [ ]:
data_era5.ta.plot()

In [ ]:
data_era5land = era5.add(
    data=grid,
    dataset=ERA5Dataset.ERA5LAND,
    path=os.path.join(output, "ERA5_data"),
)

In [ ]:
data_era5land.data_vars

In [ ]:
data_era5land.rsd_era5land.plot()

In [ ]:
data_era5land.ta.plot()

In [ ]:
data_mixte = era5.add(
    data=grid,
    dataset=ERA5Dataset.ERA5LAND,
    variables=[ERA5Var.TEMPERATURE, ERA5Var.DEWPOINT_TEMPERATURE],
    path=os.path.join(output, "ERA5_data"),
)
data_mixte = era5.add(
    data=data_mixte,
    dataset=ERA5Dataset.ERA5,
    variables=[
        ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD_CLEAR_SKY,
        ERA5Var.SURFACE_THERMAL_RADIATION_DOWNWARD_CLEAR_SKY,
    ],
    path=os.path.join(output, "ERA5_data"),
)

In [ ]:
data_mixte.data_vars

In [ ]:
# with error
try:
    data_error = era5.add(
        data=grid,
        dataset=ERA5Dataset.ERA5LAND,
        variables=[
            ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD_CLEAR_SKY,
            ERA5Var.SURFACE_THERMAL_RADIATION_DOWNWARD_CLEAR_SKY,
        ],
        path=os.path.join(output, "ERA5_data"),
    )
except KeyError:
    print("Error")

In [ ]:
# with error
data_rsd = era5.add(
    data=grid,
    dataset=ERA5Dataset.ERA5,
    variables=[
        ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD,
    ],
    path=os.path.join(output, "ERA5_data"),
)
data_rsd_cs = era5.add(
    data=grid,
    dataset=ERA5Dataset.ERA5,
    variables=[
        ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD_CLEAR_SKY,
    ],
    path=os.path.join(output, "ERA5_data"),
)
diff = data_rsd_cs["rsd_era5"] - data_rsd["rsd_era5"]
diff.plot()

# Get DEM

In [ ]:
# ROI
roi_path = os.path.join("data", "Zone_Senegal_Centre.shp")
roi_bbox, roi_crs = get_utm_bbox_from_roi(roi_path)

In [ ]:
mnt_path = os.path.join(os.environ["MNT_PATH"], "DEM_Copernicus_30m/")
data = get_dem_from_roi(
    roi_bbox=roi_bbox, roi_crs=roi_crs, base_dir=mnt_path, resolution=100
)

In [ ]:
# Add attributes
data.attrs["vis_date"] = date.date()
data.attrs["vis_time"] = date.time()
data.attrs["tir_date"] = date.date()
data.attrs["tir_time"] = date.time()
data

In [ ]:
plot_dem(data)

In [ ]:
updated_data = era5.add(
    data=data,
    dataset=ERA5Dataset.ERA5LAND,
    variables=[ERA5Var.TEMPERATURE, ERA5Var.DEWPOINT_TEMPERATURE],
    path=os.path.join(output, "ERA5_data"),
)
updated_data = era5.add(
    data=updated_data,
    dataset=ERA5Dataset.ERA5,
    variables=[
        ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD_CLEAR_SKY,
        ERA5Var.SURFACE_THERMAL_RADIATION_DOWNWARD_CLEAR_SKY,
    ],
    path=os.path.join(output, "ERA5_data"),
)

In [ ]:
plot_dataset(updated_data)

# Temperature rescaling

Add temperature and rescale it using the selected method and interpolation

In [ ]:
updated_data = era5.add(
    data=data,
    dataset=ERA5Dataset.ERA5,
    variables=[ERA5Var.TEMPERATURE, ERA5Var.DEWPOINT_TEMPERATURE],
    temp_method=RescalTempMethod.MONTHLY_LR,
    interp_type=rio.enums.Resampling.cubic_spline,
)


In [ ]:
plot_dataset(updated_data)

In [ ]:
era5_surface, era5_pressure, era5_dem, era5_dem_pressure = prepare_data(
    data=data,
    dataset=ERA5Dataset.ERA5,
    variables=[
        ERA5Var.TEMPERATURE,
        ERA5Var.DEWPOINT_TEMPERATURE,
        ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD_CLEAR_SKY,
        ERA5Var.SURFACE_THERMAL_RADIATION_DOWNWARD_CLEAR_SKY,
    ],
    temp_method=RescalTempMethod.LR_PROFILE_ALT_DEP,
)


In [ ]:
era5_surface

In [ ]:
era5_pressure

In [ ]:
era5_dem_pressure

In [ ]:
era5_dem

In [ ]:
era5_surface["ssrdc"].plot()